In [18]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../')

from sqlalchemy import create_engine, text

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

In [19]:
#path_xlsx_avance = r'G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/AVANCE DE COSECHA V1.xlsx'
path_xlsx_avance = r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\AVANCE DE COSECHA.xlsx'

In [20]:
xlsx_resumen = pd.read_excel(path_xlsx_avance, sheet_name='AVANCE_FECHAS')

In [21]:
xlsx_resumen

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-06-05 00:00:00,1.560303,139.04,NaN
1,1,(blank),583.665310,0.00,NaN
2,2,(blank),200.798730,0.00,NaN
3,3,(blank),102.999873,0.00,NaN
4,4,2026-05-29 00:00:00,0.974739,72.12,86.35
...,...,...,...,...,...
1010,668,(blank),60.368766,0.00,NaN
1011,669,(blank),16.606813,0.00,NaN
1012,(blank),2026-06-20 00:00:00,5.668316,229.54,286.68
1013,(blank),(blank),90.880924,0.00,NaN


In [22]:
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')

In [24]:
# Convertir a número (los no válidos se vuelven NaN)
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')
# Eliminar filas con NaN
xlsx_resumen = xlsx_resumen.dropna(subset=['COD_COS'])
# Convertir a int
xlsx_resumen['COD_COS'] = xlsx_resumen['COD_COS'].astype(int)
xlsx_resumen

C:\Users\bismarksr\AppData\Local\Temp\ipykernel_31724\3178366617.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')


,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-06-05 00:00:00,1.560303,139.04,NaN
1,1,(blank),583.665310,0.00,NaN
2,2,(blank),200.798730,0.00,NaN
3,3,(blank),102.999873,0.00,NaN
4,4,2026-05-29 00:00:00,0.974739,72.12,86.35
...,...,...,...,...,...
1007,666,(blank),844.338401,0.00,NaN
1008,667,(blank),8.853639,0.00,NaN
1009,668,2026-06-20 00:00:00,NaN,NaN,293.04
1010,668,(blank),60.368766,0.00,NaN


In [25]:
# 1. Intentar convertir a fecha (los no válidos quedan como NaT)
xlsx_resumen['FECHA'] = pd.to_datetime(xlsx_resumen['FECHA'], errors='coerce')
# Eliminar filas con fechas inválidas
xlsx_resumen = xlsx_resumen.dropna(subset=['FECHA'])

In [6]:
xlsx_resumen[xlsx_resumen['COD_COS'] == 45]

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
881,45,2025-06-13,NaN,NaN,25.81
882,45,2025-06-16,NaN,NaN,139.89
883,45,2025-06-18,NaN,NaN,139.78
884,45,2025-06-22,6.03,506.17,304.08
885,45,2025-08-02,1.11,72.00,1548.56
886,45,2025-08-09,68.58,5765.18,6003.34
887,45,2025-08-11,22.27,1726.55,1791.97
888,45,2025-08-14,20.38,1615.68,2183.57
889,45,2025-08-19,31.95,2915.56,4674.00
890,45,2025-08-27,43.85,4071.14,7255.71


In [26]:
def colapsar_tabla(df_param):
    suma_acumulador = [None] * len(df_param)
    acumulador = 0
    contador = 0
    for i, r in df_param.iterrows():
        if(r['AREA'] == 0):
            acumulador += r['ENTREGADO']
            contador += 1
        else:
            acumulador += r['ENTREGADO']
            suma_acumulador[contador] = acumulador
            acumulador = 0
            contador += 1
            continue
    if suma_acumulador[-1] == None:
        suma_acumulador[-1] = acumulador

    df_param['ENTREGAS'] = suma_acumulador
    
    df_param = df_param.dropna(subset=['ENTREGAS'])
    df_param = df_param.drop(columns=['ENTREGADO'])
    #df_param = df_param[df_param['AREA']!=0]
    return df_param

In [27]:
list_cod_cos = list(set(xlsx_resumen['COD_COS']))

In [28]:
list_df = []

In [29]:
for i in list_cod_cos:
    grupo = xlsx_resumen[xlsx_resumen['COD_COS']==i].fillna(0)
    result = colapsar_tabla(grupo)
    list_df.append(result)

In [30]:
df_combinado = pd.concat(list_df, ignore_index=True)

In [31]:
df_combinado[df_combinado['COD_COS'] == 45]

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS


In [33]:
df_combinado.to_excel(r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')
#df_combinado.to_excel('G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')

In [34]:
df_combinado.head(2)

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
0,1,2026-06-05,1.560303,139.04,0.00
1,4,2026-05-29,0.974739,72.12,86.35


In [35]:
def cargar_avance_fechas_a_db(df):
    df = df.rename(columns={
            'COD_COS': 'cod_cos',
            'FECHA': 'fecha',
            'AREA': 'area',
            'ESTIMADO': 'estimado',
            'ENTREGAS': 'entregas'
    })
    
    df['cod_cos']     = df['cod_cos'].astype('Int64')
    #df['fecha']   = pd.to_datetime(df["fecha"])
    df['area']  = df['area'].astype('float')
    df['estimado']   = df['estimado'].astype('float')
    df['entregas']   = df['entregas'].astype('float')

    #validad divicion entre 0
    df["tch_estimado"] = np.where(df["area"] > 0,
                              df["estimado"] / df["area"],
                              0)
    #validad divicion entre 0
    df["tch_entregas"] = np.where(df["area"] > 0,
                              df["entregas"] / df["area"],
                              0)
    
    df['diferencia_tn'] = df['entregas'] - df['estimado']
    df["porcen_dif"] = np.where(df["entregas"] > 0,
                              df["diferencia_tn"] / df["entregas"],
                              0)

    with ENGINE.begin() as conn:  # Inicia transacción
        # Vaciar la tabla y reiniciar secuencia
        conn.execute(text(f'TRUNCATE TABLE catastro_iag.data_avance_fechas RESTART IDENTITY'))
        
        # Insertar nuevos datos
        df.to_sql(
            name='data_avance_fechas',
            con=conn,  # conexión cruda dentro de la transacción
            schema='catastro_iag',
            if_exists='append',
            index=False,
            method='multi',
            chunksize=1000
        )
    print(f"✅ Se ha actualziado la tabla de codigos cosecha")

In [36]:
cargar_avance_fechas_a_db(df_combinado)

✅ Se ha actualziado la tabla de codigos cosecha
